# NB94: The Dilution Factor — Analytic Mass Prediction

## Motivation

NB93 established three facts about the lepton third generation:
1. **All CP asymmetry resides in window 0** (#211) — the first 48 coprime crossings
2. **R₃ is analytically linear** (#212) — slope matches exp(−κt) to machine precision
3. **The cumulative R₃ ratio dilutes monotonically** with window count

The mass prediction m_τ/m_μ = R₃^{x₃} depends on the number of windows included.
At window-0 only: +13.7%. At T=5000 (~24 windows): −5.12%.

NB74 (#148) established the **dilution model**:
$$R^2(n) = \frac{\sigma_1^2 + (n-1)\sigma_\infty^2}{\sigma_2^2 + (n-1)\sigma_\infty^2}$$

where σ₁, σ₂ are the window-0 CP-partner RMS values and σ_∞ is the late-time
equalized RMS. The question: **what value of n is the physical prediction?**

## Strategy

1. Extract σ₁, σ₂, σ_∞ for both R₃ and R₄ lepton channels from all 210 branches
2. Verify the dilution model matches ODE cumulative ratios
3. Solve for n where R₃^{x₃} = m_τ/m_μ and R₄^{x₄_lep} = m_μ/m_e
4. Check if these n values coincide → single physical window count
5. Scan algebraic candidates for n_phys

## Identity targets: #213+
Running total entering: 212 identities, 0 free parameters

In [1]:
# -- NB94 Setup --
import sys, numpy as np, time
from pathlib import Path
from math import gcd
from fractions import Fraction

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               X4, X3, X2, LAM7, X4_LEP,
                               PHYSICAL_CROSSINGS, CP_PAIRS, SM_TARGETS,
                               ACTIVE_PRIMES)
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

print(f"Device: {detect_device()}")
jax_warmup()

sys0 = SolenoidSystem()
P = SA.P        # 210
PHI = SA.PHI    # 48
PRIMES = SA.primes
WINDOW_SIZE = PHI  # 48 coprime crossings per 210-period

# SM targets (PDG)
M_MU_OVER_M_E = SM_TARGETS['m_mu/m_e'][0]   # 206.768
M_TAU_OVER_M_MU = 16.817                     # m_tau/m_mu
M_TAU_OVER_M_E = 3477.2                      # m_tau/m_e

# Lepton CP pair
cp_lepton = CP_PAIRS['LEPTON']  # (a3=0, a7_g1=1, a7_g2=5)
a3_lep, a7_g1, a7_g2 = cp_lepton

# Quark CP pair
cp_quark = CP_PAIRS['QUARK']  # (a3=1, a7_g1=4, a7_g2=2)
a3_qrk, a7_g1_q, a7_g2_q = cp_quark

# Integration at T=20000 for full dilution curve (~95 windows)
T_max = 20000
cis = SA.coprime_indices(T_max)

# TIME CONVENTION: crossing ci happens at time t = ci
# (θ₀(ci) = 2π·ci = ci-th base revolution; CRT labels depend on ci mod {3,5,7})
# Previous NB94 used t = ci + 1 (off-by-one error from NB81).
# NB95 showed this changes σ∞ values and the √30 ratio.
# We now test BOTH conventions to determine which is correct.

# Convention A: t = ci (correct: ci-th crossing at time ci)
t_cross_A = cis.astype(float)
T_integ_A = float(T_max + 1)

# Convention B: t = ci + 1 (NB81 legacy offset)
t_cross_B = (cis + 1).astype(float)
T_integ_B = float(T_max + 2)

# CRT labels for each crossing (independent of time convention)
a3_t, a5_t, a7_t = SA.sector_labels(cis)

branches = sys0.all_branches()
print(f"Integrating {len(branches)} branches to T={T_max} with BOTH conventions...")

t0 = time.time()
res_A = sys0.integrate_all_branches(branches, t_cross_A, T_integ_A, backend='jax')
t1 = time.time()
res_B = sys0.integrate_all_branches(branches, t_cross_B, T_integ_B, backend='jax')
t2 = time.time()

n_cross = len(cis)
n_windows = n_cross // WINDOW_SIZE
print(f"Convention A (t=ci):   {t1-t0:.1f}s")
print(f"Convention B (t=ci+1): {t2-t1:.1f}s")
print(f"{n_cross} coprime crossings, {n_windows} full windows")
print(f"Lepton CP pair: a3={a3_lep}, a7_g1={a7_g1}, a7_g2={a7_g2}")
print(f"Quark  CP pair: a3={a3_qrk}, a7_g1={a7_g1_q}, a7_g2={a7_g2_q}")
print(f"Exponents: x3={X3:.4f}, x4_lep={X4_LEP:.4f}, x4={X4:.4f}")
print(f"SM targets: m_mu/m_e={M_MU_OVER_M_E:.3f}, m_tau/m_mu={M_TAU_OVER_M_MU:.3f}")

Device: CPU (1 device(s))
Integrating 210 branches to T=20000 with BOTH conventions...
  JAX [CPU (1 device(s))]: 210 branches, 4572 eval pts, T=20001.0 — 133.59s
  JAX [CPU (1 device(s))]: 210 branches, 4572 eval pts, T=20002.0 — 137.90s
Convention A (t=ci):   133.6s
Convention B (t=ci+1): 137.9s
4572 coprime crossings, 95 full windows
Lepton CP pair: a3=0, a7_g1=1, a7_g2=5
Quark  CP pair: a3=1, a7_g1=4, a7_g2=2
Exponents: x3=1.9099, x4_lep=7.7986, x4=7.6394
SM targets: m_mu/m_e=206.768, m_tau/m_mu=16.817


## Section 1: Per-Window RMS Extraction

For each window $w$ of 48 coprime crossings, extract the mean squared R value
at R₃ and R₄ for the lepton CP partners (a₇=1 and a₇=5, with a₃=0).

This gives us:
- $M_{k,g}(w)$ = mean over branches of $R_k^2$ at crossings where $a_7 = g$ (within lepton sector)
- In window 0: $M_{k,1}(0) \neq M_{k,5}(0)$ (CP asymmetry)
- In windows $w \geq 1$: $M_{k,1}(w) = M_{k,5}(w) = M_{k,\infty}$ (equalized)

From these, the dilution model parameters are:
$$\sigma_1^2 = M_{k,g_1}(0), \quad \sigma_2^2 = M_{k,g_2}(0), \quad \sigma_\infty^2 = M_{k,\infty}$$

In [2]:
# -- Section 1: Per-window RMS for BOTH conventions --
# Convention A: t = ci (exact crossing time)
# Convention B: t = ci + 1 (NB81 legacy offset)
# Compare dilution parameters side-by-side.

n_full_windows = n_cross // WINDOW_SIZE

def extract_per_window_rms(res_dict, cp_pair):
    """Extract per-window sector RMS for a CP pair from integration results."""
    a3_cp, a7_cp_g1, a7_cp_g2 = cp_pair
    branch_list = list(res_dict.keys())
    wrms = {'R3': {'g1': [], 'g2': []}, 'R4': {'g1': [], 'g2': []}}
    for w in range(n_full_windows):
        i0 = w * WINDOW_SIZE
        i1 = i0 + WINDOW_SIZE
        win_cis = cis[i0:i1]
        win_res = {b: res_dict[b][i0:i1, :] for b in branch_list}
        sec_rms = sys0.accumulate_sectors(
            win_res, win_cis, a3_t[i0:i1], a5_t[i0:i1], a7_t[i0:i1]
        )
        wrms['R3']['g1'].append(sec_rms[0, a3_cp, a7_cp_g1, 2])
        wrms['R3']['g2'].append(sec_rms[0, a3_cp, a7_cp_g2, 2])
        wrms['R4']['g1'].append(sec_rms[0, a3_cp, a7_cp_g1, 3])
        wrms['R4']['g2'].append(sec_rms[0, a3_cp, a7_cp_g2, 3])
    for k in wrms:
        for g in wrms[k]:
            wrms[k][g] = np.array(wrms[k][g])
    return wrms

def extract_params(wrms):
    """Extract dilution model parameters from per-window RMS."""
    prm = {}
    for lev in ['R3', 'R4']:
        g1, g2 = wrms[lev]['g1'], wrms[lev]['g2']
        prm[lev] = {
            'sigma1_sq': g1[0]**2,
            'sigma2_sq': g2[0]**2,
            'sigma_inf_sq': np.mean(np.concatenate([g1[2:]**2, g2[2:]**2])),
            'R0': g1[0] / g2[0],
        }
    return prm

# Extract for both conventions, lepton sector
win_rms_A = extract_per_window_rms(res_A, cp_lepton)
win_rms_B = extract_per_window_rms(res_B, cp_lepton)
params_A = extract_params(win_rms_A)
params_B = extract_params(win_rms_B)

# Display side-by-side comparison
print('CONVENTION COMPARISON: LEPTON SECTOR (a3=0, a5=0)')
print('=' * 100)
print(f'  Convention A: t = ci (exact crossing time)')
print(f'  Convention B: t = ci + 1 (NB81 legacy offset)')
print()

for lev in ['R3', 'R4']:
    pA, pB = params_A[lev], params_B[lev]
    x = X3 if lev == 'R3' else X4_LEP
    tgt = M_TAU_OVER_M_MU if lev == 'R3' else M_MU_OVER_M_E
    print(f'{lev}:')
    print(f'  {"Parameter":<20} {"Conv A (t=ci)":>16} {"Conv B (t=ci+1)":>16}')
    print(f'  {"-"*54}')
    print(f'  {"sigma1":.<20} {np.sqrt(pA["sigma1_sq"]):>16.8f} {np.sqrt(pB["sigma1_sq"]):>16.8f}')
    print(f'  {"sigma2":.<20} {np.sqrt(pA["sigma2_sq"]):>16.8f} {np.sqrt(pB["sigma2_sq"]):>16.8f}')
    print(f'  {"sigma_inf":.<20} {np.sqrt(pA["sigma_inf_sq"]):>16.8f} {np.sqrt(pB["sigma_inf_sq"]):>16.8f}')
    print(f'  {"R0 = s1/s2":.<20} {pA["R0"]:>16.8f} {pB["R0"]:>16.8f}')
    print(f'  {"R0^x (mass pred)":.<20} {pA["R0"]**x:>16.4f} {pB["R0"]**x:>16.4f}')
    print(f'  {"SM target":.<20} {tgt:>16.3f} {tgt:>16.3f}')
    print()

# sigma_inf ratio R4/R3
si_A3 = np.sqrt(params_A['R3']['sigma_inf_sq'])
si_A4 = np.sqrt(params_A['R4']['sigma_inf_sq'])
si_B3 = np.sqrt(params_B['R3']['sigma_inf_sq'])
si_B4 = np.sqrt(params_B['R4']['sigma_inf_sq'])
ratio_A = si_A4 / si_A3
ratio_B = si_B4 / si_B3
print(f'sigma_inf(R4)/sigma_inf(R3):')
print(f'  Conv A: {ratio_A:.6f}  (sqrt(30) = {np.sqrt(30):.6f}, dev = {(ratio_A/np.sqrt(30)-1)*100:+.4f}%)')
print(f'  Conv B: {ratio_B:.6f}  (sqrt(30) = {np.sqrt(30):.6f}, dev = {(ratio_B/np.sqrt(30)-1)*100:+.4f}%)')

# Window-0 CP ratios (the actual mass-prediction inputs)
print(f'\nWINDOW-0 CP RATIOS (mass prediction inputs):')
for lev in ['R3', 'R4']:
    R0_A = params_A[lev]['R0']
    R0_B = params_B[lev]['R0']
    print(f'  {lev}: R0_A = {R0_A:.8f}, R0_B = {R0_B:.8f}, diff = {(R0_A/R0_B-1)*100:+.4f}%')

# Use Convention A as primary, keep B for reference
params = params_A
win_rms = win_rms_A
res = res_A
print(f'\n>>> Using Convention A (t=ci) as PRIMARY for all downstream analysis <<<')
print(f'>>> Convention B retained in params_B/win_rms_B for comparison <<<')

CONVENTION COMPARISON: LEPTON SECTOR (a3=0, a5=0)
  Convention A: t = ci (exact crossing time)
  Convention B: t = ci + 1 (NB81 legacy offset)

R3:
  Parameter               Conv A (t=ci)  Conv B (t=ci+1)
  ------------------------------------------------------
  sigma1..............       2.08968869       2.10006315
  sigma2..............       0.39976481       0.44788323
  sigma_inf...........       0.03642061       0.04391884
  R0 = s1/s2..........       5.22729530       4.68886307
  R0^x (mass pred)....          23.5401          19.1269
  SM target...........           16.817           16.817

R4:
  Parameter               Conv A (t=ci)  Conv B (t=ci+1)
  ------------------------------------------------------
  sigma1..............       1.97360050       2.02194315
  sigma2..............       0.33383215       0.32732802
  sigma_inf...........       0.26613490       0.24038254
  R0 = s1/s2..........       5.91195458       6.17711596
  R0^x (mass pred)....     1043316.4625     14689

In [3]:
# Write comparison summary to temp file
with open(ROOT / 'temp' / 'nb94_params.txt', 'w') as f:
    f.write('CONVENTION COMPARISON\n')
    f.write('=' * 60 + '\n')
    for conv_label, prm in [('A (t=ci)', params_A), ('B (t=ci+1)', params_B)]:
        f.write(f'\nConvention {conv_label}:\n')
        for lev in ['R3', 'R4']:
            p = prm[lev]
            f.write(f'  {lev}: s1={np.sqrt(p["sigma1_sq"]):.8f}, '
                    f's2={np.sqrt(p["sigma2_sq"]):.8f}, '
                    f'sinf={np.sqrt(p["sigma_inf_sq"]):.8f}, '
                    f'R0={p["R0"]:.8f}\n')
        si3 = np.sqrt(prm['R3']['sigma_inf_sq'])
        si4 = np.sqrt(prm['R4']['sigma_inf_sq'])
        f.write(f'  sinf ratio R4/R3 = {si4/si3:.6f} (sqrt(30)={np.sqrt(30):.6f})\n')
print('Written to temp/nb94_params.txt')

Written to temp/nb94_params.txt


## Section 2: Dilution Model Validation

The dilution model from NB74 (#148):
$$R^2(n) = \frac{\sigma_1^2 + (n-1)\sigma_\infty^2}{\sigma_2^2 + (n-1)\sigma_\infty^2}$$

Verify this matches the cumulative R ratios computed directly from the ODE data.
Then solve for the physical window count $n$ where:
- $R_3(n)^{x_3} = m_\tau/m_\mu = 16.817$
- $R_4(n)^{x_{4,\ell}} = m_\mu/m_e = 206.768$

If these two equations yield the **same** $n$, that $n$ is the physical dilution factor.

In [4]:
# -- Section 2: Dilution model validation — BOTH conventions --

def R_model(n, sigma1_sq, sigma2_sq, sigma_inf_sq):
    """Dilution model: R(n) = sqrt((s1^2 + (n-1)*sinf^2) / (s2^2 + (n-1)*sinf^2))"""
    return np.sqrt((sigma1_sq + (n - 1) * sigma_inf_sq) /
                   (sigma2_sq + (n - 1) * sigma_inf_sq))

def compute_cumulative_R(res_dict):
    """Compute cumulative CP ratio at each window count."""
    branch_list = list(res_dict.keys())
    cum = {'R3': [], 'R4': []}
    for n_w in range(1, n_full_windows + 1):
        end_idx = n_w * WINDOW_SIZE
        cum_cis = cis[:end_idx]
        cum_res = {b: res_dict[b][:end_idx, :] for b in branch_list}
        sec_rms = sys0.accumulate_sectors(
            cum_res, cum_cis, a3_t[:end_idx], a5_t[:end_idx], a7_t[:end_idx]
        )
        cp = sys0.cp_pair_ratios(sec_rms)
        cum['R3'].append(cp['LEPTON'][2])
        cum['R4'].append(cp['LEPTON'][3])
    for k in cum:
        cum[k] = np.array(cum[k])
    return cum

n_range = np.arange(1, n_full_windows + 1)

# Compute cumulative R for both conventions
cum_R_A = compute_cumulative_R(res_A)
cum_R_B = compute_cumulative_R(res_B)

# Use convention A as primary
cum_R = cum_R_A

# Compare model fit for both conventions
print('DILUTION MODEL FIT: Convention A vs B')
print('=' * 100)
for conv_label, prm, cum in [('A (t=ci)', params_A, cum_R_A), ('B (t=ci+1)', params_B, cum_R_B)]:
    print(f'\n--- Convention {conv_label} ---')
    for lev in ['R3', 'R4']:
        x = X3 if lev == 'R3' else X4_LEP
        tgt = M_TAU_OVER_M_MU if lev == 'R3' else M_MU_OVER_M_E
        p = prm[lev]
        mod = R_model(n_range, p['sigma1_sq'], p['sigma2_sq'], p['sigma_inf_sq'])
        resids = np.abs(mod / cum[lev] - 1) * 100
        print(f'  {lev}: max model residual = {np.max(resids):.4f}%')
        print(f'  {"n":>6}  {"ODE":>10}  {"Model":>10}  {"Resid":>8}  {"Mass":>10}  {"Dev":>7}')
        for nw in [1, 5, 10, 15, 17, 20, 24, 30, 50, 70, 95]:
            if nw > n_full_windows: continue
            i = nw - 1
            ode_v = cum[lev][i]
            mod_v = mod[i]
            mass = ode_v ** x
            dev = (mass / tgt - 1) * 100
            m = ' <--' if nw == 17 else ''
            print(f'  {nw:>6d}  {ode_v:>10.6f}  {mod_v:>10.6f}  {(mod_v/ode_v-1)*100:>+7.4f}%  {mass:>10.4f}  {dev:>+6.2f}%{m}')

# Solve for n_phys under BOTH conventions
from scipy.optimize import brentq

print('\n' + '=' * 100)
print('n_phys COMPARISON: Convention A vs Convention B')
print('=' * 100)

n_results = {}
for conv_label, prm, cum in [('A', params_A, cum_R_A), ('B', params_B, cum_R_B)]:
    n_results[conv_label] = {}
    print(f'\n--- Convention {conv_label} ---')
    for lev, x, tgt, label in [
        ('R3', X3, M_TAU_OVER_M_MU, 'm_tau/m_mu'),
        ('R4', X4_LEP, M_MU_OVER_M_E, 'm_mu/m_e'),
    ]:
        p = prm[lev]
        R_target = tgt ** (1.0 / x)
        R_at_1 = R_model(1, p['sigma1_sq'], p['sigma2_sq'], p['sigma_inf_sq'])
        if R_target > R_at_1:
            print(f'  {lev} ({label}): NO SOLUTION (target > R(1))')
            n_results[conv_label][lev] = None
        else:
            def obj(n, s1=p['sigma1_sq'], s2=p['sigma2_sq'], si=p['sigma_inf_sq']):
                return R_model(n, s1, s2, si) - R_target
            n_sol = brentq(obj, 1.0, 1e8)
            R_chk = R_model(n_sol, p['sigma1_sq'], p['sigma2_sq'], p['sigma_inf_sq'])
            print(f'  {lev} ({label}): n_phys = {n_sol:.4f}  (R={R_chk:.6f}, mass={R_chk**x:.4f})')
            n_results[conv_label][lev] = n_sol

# Key comparison: at n=17 (sum of primes), what do both conventions give?
print('\n' + '=' * 100)
print('MASS PREDICTIONS AT n=17 = sum(primes): Convention A vs B')
print('=' * 100)
for conv_label, cum in [('A (t=ci)', cum_R_A), ('B (t=ci+1)', cum_R_B)]:
    r3 = cum['R3'][16]
    r4 = cum['R4'][16]
    m_tm = r3 ** X3
    m_me = r4 ** X4_LEP
    d_tm = (m_tm / M_TAU_OVER_M_MU - 1) * 100
    d_me = (m_me / M_MU_OVER_M_E - 1) * 100
    print(f'  Conv {conv_label}:  m_tau/m_mu = {m_tm:.4f} ({d_tm:+.2f}%)  m_mu/m_e = {m_me:.4f} ({d_me:+.2f}%)')

# Write to temp file
with open(ROOT / 'temp' / 'nb94_section2.txt', 'w') as f:
    for conv_label in ['A', 'B']:
        f.write(f'\nConvention {conv_label}:\n')
        for lev in ['R3', 'R4']:
            n_val = n_results[conv_label][lev]
            f.write(f'  n_phys({lev}) = {n_val}\n')
    f.write(f'\nAt n=17:\n')
    for conv_label, cum in [('A', cum_R_A), ('B', cum_R_B)]:
        r3 = cum['R3'][16]
        r4 = cum['R4'][16]
        f.write(f'  Conv {conv_label}: R3^x3={r3**X3:.4f}, R4^x4l={r4**X4_LEP:.4f}\n')
print('\nWritten to temp/nb94_section2.txt')

DILUTION MODEL FIT: Convention A vs B

--- Convention A (t=ci) ---
  R3: max model residual = 0.0000%
       n         ODE       Model     Resid        Mass      Dev
       1    5.227295    5.227295  +0.0000%     23.5401  +39.98%
       5    5.145748    5.145747  -0.0000%     22.8437  +35.84%
      10    5.049241    5.049240  -0.0000%     22.0325  +31.01%
      15    4.958236    4.958235  -0.0000%     21.2803  +26.54%
      17    4.923261    4.923260  -0.0000%     20.9945  +24.84% <--
      20    4.872235    4.872234  -0.0000%     20.5809  +22.38%
      24    4.806741    4.806740  -0.0000%     20.0557  +19.26%
      30    4.713543    4.713542  -0.0000%     19.3196  +14.88%
      50    4.440006    4.440005  -0.0000%     17.2350   +2.49%
      70    4.211696    4.211696  -0.0000%     15.5821   -7.34%
      95    3.973329    3.973328  -0.0000%     13.9412  -17.10%
  R4: max model residual = 0.0002%
       n         ODE       Model     Resid        Mass      Dev
       1    5.911955    5.9

## Section 3: Convention-Independent Reformulation

The +1 time offset from NB81 was arbitrary — it shifts the sampling phase by
$2\pi/P_k$ per level. Since late-time residuals are pure sinusoids at primorial
frequencies, the sampled σ_∞ = |A_k sin(phase)| depends on the convention.

**The fix**: NB95 proved the late-time spectrum is analytic:
$$R_k(t) \approx A_k \sin(\omega/P_k \cdot t + \phi_k), \quad A_k = \frac{\varepsilon}{\sqrt{\kappa^2 + (\omega/P_k)^2}}$$

The convention-independent σ_∞ is the **time-averaged RMS**:
$$\sigma_{\infty,k} = A_k / \sqrt{2}$$

This eliminates all phase dependence. The dilution model becomes:
$$R^2(n) = \frac{\sigma_1^2 + (n-1) A_k^2/2}{\sigma_2^2 + (n-1) A_k^2/2}$$

where σ₁, σ₂ come from window-0 (convention-dependent, but window-0 is in the
transient regime where the specific crossing time matters physically).

In [6]:
# -- Section 3: Convention-independent reformulation --

# A. Analytic amplitudes from NB95 formula
# Level k has dominant frequency omega/P_k where P_k = k-th primorial
# P_0=1, P_1=2, P_2=6, P_3=30  (NB95 confirmed)
# A_k = epsilon / sqrt(kappa^2 + (omega/P_k)^2)
# sigma_inf_k = A_k / sqrt(2)  (time-averaged RMS of sinusoid)
primorials = [1, 2, 6, 30, 210]

print('ANALYTIC LATE-TIME AMPLITUDES')
print('=' * 80)
print(f'epsilon = kappa = 1/sqrt(210) = {EPSILON:.8f}')
print(f'omega = 2*pi = {OMEGA:.6f}')
print()
print(f'{"Level":<6} {"P_k":<6} {"omega/P_k":>12} {"A_k":>14} {"sigma_inf":>14}')
print('-' * 56)
A_analytic = []
sigma_inf_analytic = []
for k in range(4):
    Pk = primorials[k]  # P_0=1, P_1=2, P_2=6, P_3=30
    omega_k = OMEGA / Pk
    Ak = EPSILON / np.sqrt(KAPPA**2 + omega_k**2)
    sk = Ak / np.sqrt(2)
    A_analytic.append(Ak)
    sigma_inf_analytic.append(sk)
    print(f'R_{k:<4} {Pk:<6} {omega_k:>12.6f} {Ak:>14.8f} {sk:>14.8f}')

# NB95 measured values for cross-check
nb95_measured = [0.007765, 0.016007, 0.046782, 0.221463]
print(f'\nCross-check vs NB95 measured sigma_inf (continuous-time RMS):')
for k in range(4):
    dev = (sigma_inf_analytic[k] / nb95_measured[k] - 1) * 100
    print(f'  R_{k}: analytic={sigma_inf_analytic[k]:.6f}, NB95={nb95_measured[k]:.6f}, dev={dev:+.2f}%')

# B. Compare analytic sigma_inf with BOTH sampled conventions
# 'R3' in our dict = cascade level 2 (P_2=6)
# 'R4' in our dict = cascade level 3 (P_3=30)
print(f'\nCOMPARISON: analytic vs sampled sigma_inf')
print('=' * 80)
print(f'{"Level":<10} {"Analytic A/√2":>14} {"Conv A":>14} {"Conv B":>14} {"dev A":>8} {"dev B":>8}')
print('-' * 68)
level_map = {'R3': 2, 'R4': 3}
for lev, lev_idx in level_map.items():
    s_ana = sigma_inf_analytic[lev_idx]
    s_A = np.sqrt(params_A[lev]['sigma_inf_sq'])
    s_B = np.sqrt(params_B[lev]['sigma_inf_sq'])
    dev_A = (s_A / s_ana - 1) * 100
    dev_B = (s_B / s_ana - 1) * 100
    print(f'{lev} (lev {lev_idx})'
          f' {s_ana:>14.8f} {s_A:>14.8f} {s_B:>14.8f} {dev_A:>+7.2f}% {dev_B:>+7.2f}%')

# C. Convention-independent sigma_inf ratio
ratio_analytic = sigma_inf_analytic[3] / sigma_inf_analytic[2]
print(f'\nCONVENTION-INDEPENDENT sigma_inf RATIO (R4/R3 = level 3/level 2):')
print(f'  Analytic: {ratio_analytic:.6f}')
print(f'  sqrt(30) = {np.sqrt(30):.6f}  dev = {(ratio_analytic/np.sqrt(30)-1)*100:+.2f}%')
print(f'  Conv A:   {si_A4/si_A3:.6f}')
print(f'  Conv B:   {si_B4/si_B3:.6f}')
print(f'  NB95:     {nb95_measured[3]/nb95_measured[2]:.6f}')
print()
print(f'  The TRUE amplitude ratio is ~{ratio_analytic:.2f}, not sqrt(30)={np.sqrt(30):.2f}.')
print(f'  sqrt(30) from Conv B was a phase-sampling artifact.')

# D. Reformulated dilution model with analytic sigma_inf
print(f'\n{"="*80}')
print('DILUTION MODEL WITH ANALYTIC sigma_inf')
print(f'{"="*80}')

# Window-0 sigma1, sigma2 are convention-dependent:
print(f'\nWindow-0 R0 ratios (convention-dependent):')
for lev in ['R3', 'R4']:
    R0A = params_A[lev]['R0']
    R0B = params_B[lev]['R0']
    print(f'  {lev}: R0_A={R0A:.6f}, R0_B={R0B:.6f}, diff={((R0A/R0B)-1)*100:+.2f}%')

# n_phys with analytic sigma_inf, for BOTH window-0 conventions
from scipy.optimize import brentq

print(f'\nn_phys WITH ANALYTIC sigma_inf:')
n_results_analytic = {}
for conv_label, prm in [('A', params_A), ('B', params_B)]:
    n_results_analytic[conv_label] = {}
    print(f'\n  Window-0 from Convention {conv_label}:')
    for lev, x, tgt, label in [
        ('R3', X3, M_TAU_OVER_M_MU, 'm_tau/m_mu'),
        ('R4', X4_LEP, M_MU_OVER_M_E, 'm_mu/m_e'),
    ]:
        lev_idx = level_map[lev]
        p = prm[lev]
        s_inf_sq_ana = sigma_inf_analytic[lev_idx]**2
        R_target = tgt ** (1.0 / x)
        R_at_1 = R_model(1, p['sigma1_sq'], p['sigma2_sq'], s_inf_sq_ana)
        if R_target > R_at_1:
            print(f'    {lev} ({label}): NO SOLUTION')
            n_results_analytic[conv_label][lev] = None
        else:
            def obj(n, s1=p['sigma1_sq'], s2=p['sigma2_sq'], si=s_inf_sq_ana):
                return R_model(n, s1, s2, si) - R_target
            n_sol = brentq(obj, 1.0, 1e8)
            R_chk = R_model(n_sol, p['sigma1_sq'], p['sigma2_sq'], s_inf_sq_ana)
            print(f'    {lev} ({label}): n_phys = {n_sol:.4f}  (mass = {R_chk**x:.4f})')
            n_results_analytic[conv_label][lev] = n_sol

# n_phys with analytic sigma_inf and cumulative R from ODE (most robust)
print(f'\n{"="*80}')
print('CUMULATIVE MASS PREDICTIONS USING ODE DATA DIRECTLY')
print(f'{"="*80}')
print(f'(No dilution model needed — just ODE cumulative CP ratios)')
print(f'\nn=17: both conventions use ODE cumulative, which IS convention-dependent:')
for conv_label, cum in [('A (t=ci)', cum_R_A), ('B (t=ci+1)', cum_R_B)]:
    r3 = cum['R3'][16]
    r4 = cum['R4'][16]
    m_tm = r3**X3; m_me = r4**X4_LEP
    d_tm = (m_tm/M_TAU_OVER_M_MU - 1)*100
    d_me = (m_me/M_MU_OVER_M_E - 1)*100
    print(f'  {conv_label}: m_tau/m_mu={m_tm:.2f} ({d_tm:+.1f}%)  m_mu/m_e={m_me:.1f} ({d_me:+.1f}%)')

# Summary
print(f'\n{"="*80}')
print('SUMMARY: Everything is convention-dependent')
print(f'{"="*80}')
print(f'1. sigma_inf can be made convention-independent via Fourier amplitudes')
print(f'   -> ratio = {ratio_analytic:.4f} (NOT sqrt(30))')
print(f'2. BUT sigma_1, sigma_2 (window-0) are still convention-dependent')
print(f'3. AND the cumulative ODE ratios are convention-dependent')
print(f'4. There is NO convention-independent mass prediction from this framework')
print(f'5. The +1 offset from NB81 was unprincipled')
print(f'6. #214 (sqrt(30) ratio) is a PHASE ARTIFACT -> downgrade to NULL')

# Write results
with open(ROOT / 'temp' / 'nb94_section3.txt', 'w') as f:
    f.write(f'Convention-independent analysis:\n')
    f.write(f'Analytic sigma_inf ratio = {ratio_analytic:.6f}\n')
    f.write(f'sqrt(30) = {np.sqrt(30):.6f} (NOT matched)\n')
    for conv_label in ['A', 'B']:
        f.write(f'\nConv {conv_label} + analytic sigma_inf:\n')
        for lev in ['R3', 'R4']:
            n_val = n_results_analytic[conv_label][lev]
            f.write(f'  n_phys({lev}) = {n_val}\n')
print('\nWritten to temp/nb94_section3.txt')

ANALYTIC LATE-TIME AMPLITUDES
epsilon = kappa = 1/sqrt(210) = 0.06900656
omega = 2*pi = 6.283185

Level  P_k       omega/P_k            A_k      sigma_inf
--------------------------------------------------------
R_0    1          6.283185     0.01098207     0.00776550
R_1    2          3.141593     0.02196017     0.01552819
R_2    6          1.047198     0.06575380     0.04649496
R_3    30         0.209440     0.31293378     0.22127760

Cross-check vs NB95 measured sigma_inf (continuous-time RMS):
  R_0: analytic=0.007765, NB95=0.007765, dev=+0.01%
  R_1: analytic=0.015528, NB95=0.016007, dev=-2.99%
  R_2: analytic=0.046495, NB95=0.046782, dev=-0.61%
  R_3: analytic=0.221278, NB95=0.221463, dev=-0.08%

COMPARISON: analytic vs sampled sigma_inf
Level       Analytic A/√2         Conv A         Conv B    dev A    dev B
--------------------------------------------------------------------
R3 (lev 2)     0.04649496     0.03642061     0.04391884  -21.67%   -5.54%
R4 (lev 3)     0.22127760    

## Section 4: Quark Sector Cross-Check (Convention A)

With Convention A established as correct (t=ci for crossing ci), verify:
1. Dilution model holds for quarks
2. σ_∞ ratio in quark sector
3. Convention A mass predictions at various n

In [7]:
# -- Section 4: Convention A quark cross-check --

# Quark RMS under Convention A (correct convention)
win_rms_q = extract_per_window_rms(res_A, cp_quark)
params_q = extract_params(win_rms_q)

print('QUARK SECTOR (Convention A, correct)')
print('=' * 80)

# sigma_inf comparison: lepton vs quark
si_L2 = np.sqrt(params_A['R3']['sigma_inf_sq'])  # R3 dict = level 2
si_L3 = np.sqrt(params_A['R4']['sigma_inf_sq'])  # R4 dict = level 3
si_Q2 = np.sqrt(params_q['R3']['sigma_inf_sq'])
si_Q3 = np.sqrt(params_q['R4']['sigma_inf_sq'])
ratio_L = si_L3 / si_L2
ratio_Q = si_Q3 / si_Q2

print(f'\nsigma_inf ratios (level 3 / level 2):')
print(f'  Lepton:   {ratio_L:.6f}')
print(f'  Quark:    {ratio_Q:.6f}')
print(f'  Analytic: {ratio_analytic:.6f}  (= A_3/A_2, convention-independent)')
print(f'  sqrt(30): {np.sqrt(30):.6f}  (NOT a property of the dynamics)')

# Window-0 CP ratios — quark sector
for lev in ['R3', 'R4']:
    p = params_q[lev]
    x = X3 if lev == 'R3' else X4
    print(f'\nQuark {lev}: R0 = {p["R0"]:.6f}, R0^x = {p["R0"]**x:.2f}')

# Quark dilution model fit
print(f'\nQuark dilution model fit (Convention A):')
branch_list = list(res_A.keys())
cum_R_q = {'R3': [], 'R4': []}
for n_w in range(1, n_full_windows + 1):
    end_idx = n_w * WINDOW_SIZE
    cum_cis = cis[:end_idx]
    cum_res = {b: res_A[b][:end_idx, :] for b in branch_list}
    sec_rms = sys0.accumulate_sectors(
        cum_res, cum_cis, a3_t[:end_idx], a5_t[:end_idx], a7_t[:end_idx]
    )
    cp = sys0.cp_pair_ratios(sec_rms)
    cum_R_q['R3'].append(cp['QUARK'][2])
    cum_R_q['R4'].append(cp['QUARK'][3])
for k in cum_R_q:
    cum_R_q[k] = np.array(cum_R_q[k])

max_resid_q = 0
for lev in ['R3', 'R4']:
    p = params_q[lev]
    mod = R_model(n_range, p['sigma1_sq'], p['sigma2_sq'], p['sigma_inf_sq'])
    resid = np.max(np.abs(mod / cum_R_q[lev] - 1)) * 100
    max_resid_q = max(max_resid_q, resid)
    print(f'  {lev}: max residual = {resid:.4f}%')

# Key mass predictions at integer n for Convention A
M_S_MD = SM_TARGETS['m_s/m_d'][0]
print(f'\nConvention A mass predictions at key n values:')
print(f'{"n":>4}  {"m_t/m_m":>10} {"dev":>7}  {"m_m/m_e":>10} {"dev":>7}  {"m_s/m_d(R4)":>12} {"dev":>7}')
print('-' * 65)
for n in [1, 5, 10, 15, 17, 18, 20, 25, 30, 40, 50, 55, 60, 70, 80, 95]:
    if n > n_full_windows: continue
    r3l = cum_R_A['R3'][n-1]
    r4l = cum_R_A['R4'][n-1]
    r4q = cum_R_q['R4'][n-1]
    m_tm = r3l**X3; m_me = r4l**X4_LEP; m_sd = r4q**X4
    d_tm = (m_tm/M_TAU_OVER_M_MU - 1)*100
    d_me = (m_me/M_MU_OVER_M_E - 1)*100
    d_sd = (m_sd/M_S_MD - 1)*100
    marker = ''
    if n == 17: marker = ' <-- sum(primes)'
    if n == 55: marker = ' <-- ~n_phys(R3)'
    print(f'{n:>4d}  {m_tm:>10.2f} {d_tm:>+6.1f}%  {m_me:>10.1f} {d_me:>+6.1f}%  {m_sd:>12.2f} {d_sd:>+6.1f}%{marker}')

QUARK SECTOR (Convention A, correct)

sigma_inf ratios (level 3 / level 2):
  Lepton:   7.307260
  Quark:    6.374520
  Analytic: 4.759174  (= A_3/A_2, convention-independent)
  sqrt(30): 5.477226  (NOT a property of the dynamics)

Quark R3: R0 = 39.801442, R0^x = 1136.53

Quark R4: R0 = 6.606742, R0^x = 1837562.11

Quark dilution model fit (Convention A):
  R3: max residual = 0.0000%
  R4: max residual = 0.0003%

Convention A mass predictions at key n values:
   n     m_t/m_m     dev     m_m/m_e     dev   m_s/m_d(R4)     dev
-----------------------------------------------------------------
   1       23.54  +40.0%   1043316.5 +504483.1%    1837562.11 +9187710.5%
   5       22.84  +35.8%      9898.0 +4687.0%       5520.33 +27501.6%
  10       22.03  +31.0%      1119.3 +441.3%        572.38 +2761.9%
  15       21.28  +26.5%       331.6  +60.4%        171.97 +759.9%
  17       20.99  +24.8%       231.4  +11.9%        121.44 +507.2% <-- sum(primes)
  18       20.85  +24.0%       197.0   -

## Section 5: Scorecard

### Findings (Corrected)

1. **#213 PASS**: Dilution model $R^2(n) = (\sigma_1^2 + (n-1)\sigma_\infty^2)/(\sigma_2^2 + (n-1)\sigma_\infty^2)$
   matches ODE data to <0.1% for both sectors and both time conventions.
   This is a structural theorem of the cascade, convention-independent.

2. **#214 NULL**: The √30 ratio was a **phase-sampling artifact** of the +1 offset
   (Convention B). Convention A (correct: t=ci for crossing ci) gives ratio 7.31.
   The convention-independent amplitude ratio is A₃/A₂ = 4.76.
   **√30 is not a property of the dynamics.**

3. **#215 NULL**: Under Convention A, n=17 gives m_τ/m_μ = 21.0 (+25%), not the
   −0.13% reported previously. The good result came from Convention B's phase error.
   **n=17 = Σ(primes) is not the physical dilution window count.**

4. **Convention A is correct**: t=ci for crossing ci. Convention B (t=ci+1) evaluates R
   at the wrong time while using CRT labels from a different revolution.
   The +1 offset from NB81 was unprincipled.

### What remains from NB94
- The dilution model structural theorem (#213) — robust, convention-independent
- Window-0 CP asymmetry is real and convention-independent in principle
- But extracting mass predictions requires knowing which phase to sample
- The framework does not yet provide a principled mass prediction for m_τ/m_μ

In [8]:
# -- Scorecard (CORRECTED) --
print('NB94 SCORECARD (CORRECTED)')
print('=' * 70)

# #213 PASS: dilution model structural theorem
print('\n#213  Dilution Model Structural Theorem')
print('      R^2(n) matches ODE to <0.1% for all levels, sectors,')
print('      conventions. Convention-independent structural result.')
max_resid = 0
for lev in ['R3', 'R4']:
    p = params_A[lev]
    mod = R_model(n_range, p['sigma1_sq'], p['sigma2_sq'], p['sigma_inf_sq'])
    resids = np.abs(mod / cum_R_A[lev] - 1) * 100
    max_resid = max(max_resid, np.max(resids))
print(f'      Max residual: {max_resid:.4f}%')
print('      Verdict: PASS')

# #214 NULL: sqrt(30) was phase artifact
print(f'\n#214  sigma_inf(R4)/sigma_inf(R3) = sqrt(30)?')
print(f'      Convention A (correct): ratio = {si_A4/si_A3:.4f} (NOT sqrt(30))')
print(f'      Convention B (+1 error): ratio = {si_B4/si_B3:.4f} (appeared as sqrt(30))')
print(f'      Amplitude ratio (conv-independent): {ratio_analytic:.4f}')
print(f'      sqrt(30) = {np.sqrt(30):.4f}')
print(f'      The sqrt(30) arose from a phase-sampling artifact.')
print(f'      Verdict: NULL — phase artifact of +1 offset')

# #215 NULL: n=17 was convention B artifact
r3a_17 = cum_R_A['R3'][16]
m_tm_a17 = r3a_17**X3
dev_a17 = (m_tm_a17/M_TAU_OVER_M_MU - 1)*100
r3b_17 = cum_R_B['R3'][16]
m_tm_b17 = r3b_17**X3
dev_b17 = (m_tm_b17/M_TAU_OVER_M_MU - 1)*100
print(f'\n#215  m_tau/m_mu at n = 17 = sum(primes)')
print(f'      Convention A (correct): {m_tm_a17:.2f} ({dev_a17:+.1f}%)')
print(f'      Convention B (+1 error): {m_tm_b17:.2f} ({dev_b17:+.1f}%)')
print(f'      The -0.13% came from the wrong convention.')
print(f'      Verdict: NULL — artifact of +1 offset')

print(f'\n{"="*70}')
print(f'Running total: 213 predictions/identities, 0 free parameters')
print(f'  #213 PASS (dilution model structural theorem)')
print(f'  #214 retracted (was CONDITIONAL, now NULL)')
print(f'  #215 retracted (was CONDITIONAL, now NULL)')
print(f'  Net change: −2 identities (215 → 213)')
print(f'')
print(f'HONEST ASSESSMENT:')
print(f'  The +1 offset from NB81 was an unprincipled error.')
print(f'  Its removal kills sqrt(30) and n=17 results.')
print(f'  The dilution model remains a valid structural theorem.')
print(f'  Mass prediction for m_tau/m_mu requires further work.')

NB94 SCORECARD (CORRECTED)

#213  Dilution Model Structural Theorem
      R^2(n) matches ODE to <0.1% for all levels, sectors,
      conventions. Convention-independent structural result.
      Max residual: 0.0002%
      Verdict: PASS

#214  sigma_inf(R4)/sigma_inf(R3) = sqrt(30)?
      Convention A (correct): ratio = 7.3073 (NOT sqrt(30))
      Convention B (+1 error): ratio = 5.4733 (appeared as sqrt(30))
      Amplitude ratio (conv-independent): 4.7592
      sqrt(30) = 5.4772
      The sqrt(30) arose from a phase-sampling artifact.
      Verdict: NULL — phase artifact of +1 offset

#215  m_tau/m_mu at n = 17 = sum(primes)
      Convention A (correct): 20.99 (+24.8%)
      Convention B (+1 error): 16.80 (-0.1%)
      The -0.13% came from the wrong convention.
      Verdict: NULL — artifact of +1 offset

Running total: 213 predictions/identities, 0 free parameters
  #213 PASS (dilution model structural theorem)
  #214 retracted (was CONDITIONAL, now NULL)
  #215 retracted (was CONDIT

In [9]:
# Summary
print(f'#213 PASS: dilution model exact (<{max_resid:.3f}%)')
print(f'#214 NULL: sqrt(30) was phase artifact of +1 offset')
print(f'#215 NULL: n=17 was phase artifact of +1 offset')
print(f'Running total: 213 identities (down from 215), 0 free parameters')

#213 PASS: dilution model exact (<0.000%)
#214 NULL: sqrt(30) was phase artifact of +1 offset
#215 NULL: n=17 was phase artifact of +1 offset
Running total: 213 identities (down from 215), 0 free parameters
